# 模块三：Python 进阶 —— 从函数到对象

---

## **本课目标**

1.  **掌握函数式编程精髓**：学会使用高阶函数、Lambda、装饰器等工具编写更优雅、更灵活的代码。
2.  **建立面向对象思维**：理解类与对象的概念，学会用面向对象的方式分析和解决问题。
3.  **掌握 OOP 三大支柱**：深入理解并应用封装、继承和多态。
4.  **编写高质量的类**：学习如何设计包含属性、方法、静态方法和运算符重载的完整类。

---

## **Part 1: 函数进阶与编程范式**

在 Python 中，函数不仅仅是代码的集合，它们是“一等公民”（First-Class Citizens）。这意味着函数可以像任何其他数据类型（如整数、列表）一样被对待：
- 可以赋值给一个变量。
-可以作为参数传递给另一个函数。
-可以作为另一个函数的返回值。

这种特性是 Python 函数式编程风格的基石，它使得代码更加灵活和强大。

### **1. 函数是“一等公民”：高阶函数**

一个接受函数作为参数，或者返回一个函数的函数，我们称之为**高阶函数 (Higher-Order Function)**。这个特性是 Python 强大灵活性的核心来源之一。

#### **回顾：我们已经见过的高阶函数 `sorted()`**

还记得我们在作业中是如何给 `sorted()` 函数传递一个 `lambda` 表达式作为 `key` 参数，来实现按自定义规则（例如按分数）排序的吗？

In [ ]:
students = [
    {'name': 'Alice', 'score': 88},
    {'name': 'Bob', 'score': 95},
    {'name': 'Charlie', 'score': 82}
]

# 按分数排序, `key` 参数接受了一个函数
sorted_students = sorted(students, key=lambda student: student['score'], reverse=True) 
print(sorted_students)

`sorted()` 就是一个典型的高阶函数，因为它接受了另一个函数作为它的参数。

#### **核心实战：构建自己的高阶函数**

理解高阶函数的最好方式，就是亲手写一个。我们的目标是编写一个通用的函数，它能处理一个列表，但具体“如何处理”由我们传进去的另一个函数决定。

In [ ]:
# 这是一个高阶函数，它接受一个列表和一个处理函数
from typing import Callable, List

def process_data(data_list: List, process_function: Callable):  # 这个地方就是说传入的是一个函数参数
    """
    遍历一个列表，并对其中每个元素应用 process_function。
    返回一个包含所有处理结果的新列表。
    """
    processed_list = []
    for item in data_list:
        # 调用我们传进来的函数
        processed_item = process_function(item)
        processed_list.append(processed_item)
    return processed_list

# --- 现在，我们定义几个不同的“处理函数” ---

def square(n):
    """计算一个数的平方"""
    return n * n

def to_uppercase(s):
    """将字符串转为大写"""
    return s.upper()

# --- 见证高阶函数的威力 ---

numbers = [1, 2, 3, 4]
words = ['hello', 'world']

# 1. 传入“平方”函数，处理数字列表
squared_numbers = process_data(numbers, square)
print(f"平方处理结果: {squared_numbers}")  # 输出: [1, 4, 9, 16]

# 2. 传入“转大写”函数，处理字符串列表
upper_words = process_data(words, to_uppercase)
print(f"大写处理结果: {upper_words}")    # 输出: ['HELLO', 'WORLD']

通过这个例子，我们可以清晰地看到，`process_data` 函数本身是通用的，它的具体行为被我们作为参数传进去的 `square` 或 `to_uppercase` 函数改变了。这就是高阶函数的核心价值：**分离“不变的逻辑”和“可变的逻辑”**。

### **2. 优雅的“匿名者”：Lambda 表达式**

当你需要的函数非常简单，只有一行表达式，并且只用一次时，`lambda` 表达式就派上了用场。它允许你快速定义一个匿名的、单行的迷你函数。

**语法**: `lambda arguments（传入参数）: expression（简单的函数体计算式）`

`lambda` 的返回值就是 `expression` 的计算结果。

In [ ]:
# 普通函数
def add(x, y):
    return x + y

# 等效的 lambda 函数
add_lambda = lambda x, y: x + y

print(add(3, 5))         # 8
print(add_lambda(3, 5))  # 8

> **重点**: 函数也像一个变量。`add` 是一个变量，存储了函数本身；而 `add(3, 5)` 是调用这个函数，获取其返回值。这与我们后面将 `lambda` 表达式直接作为参数传递是同一个道理——我们传递的是函数这个“变量”，而不是它的执行结果。

`lambda` 最大的用武之地是与高阶函数结合，避免为了一个简单的操作而专门定义一个完整的函数。例如，我们可以直接将 `lambda` 传给刚才的 `process_data` 函数：

In [ ]:
# 使用 lambda 表达式，无需预先定义 double 函数
doubled_numbers = process_data([1, 2, 3], lambda x: x * 2)
print(f"用 lambda 表达式处理的结果: {doubled_numbers}") # 输出: [2, 4, 6]

这种写法在 `sorted` 的 `key` 参数中也非常常见。

> **注意**: 不要为了用 `lambda` 而用。如果一个 `lambda` 表达式写得过长，或者逻辑稍微复杂，导致代码难以阅读，那就应该毫不犹豫地使用 `def` 定义一个完整的函数。代码的可读性通常比单行的简洁性更重要。

---

### **3. 代码的“增强器”：装饰器 (Decorator)**

**痛点**: 假设我们有几个函数，现在想为它们都增加一个功能，比如计算它们的运行时间。我们当然可以在每个函数内部修改代码，但这违反了“开放封闭原则”，而且充满了重复代码。

In [ ]:
import time

def func_a():
    start = time.time()
    print("Executing func_a...")
    time.sleep(1)
    end = time.time()
    print(f"func_a took {end - start:.2f} seconds")

def func_b():
    start = time.time()
    print("Executing func_b...")
    time.sleep(2)
    end = time.time()
    print(f"func_b took {end - start:.2f} seconds")

计时逻辑是重复的。**装饰器**就是为了解决这类问题而生的优雅方案。它本质上是一个高阶函数，接受一个函数作为输入，返回一个“增强版”的新函数。

#### **一步步构建装饰器**

1.  **核心原理：闭包 (Closure)**
    一个函数（外层函数）返回它内部定义的另一个函数（内层函数），并且内层函数引用了外层函数的变量，这就构成了闭包。这样返回出的内置函数会带着已经获取到的外层函数的变量一起进行返回

#### 闭包就类似是一种函数生成器外层函数去生成附带有信息的内层函数，外层函数就像是生成器

In [ ]:
# 闭包是什么呢
def outer():
    x = 10 # 这是外层函数的变量，在全局中正常不能访问
    def inner(): # 内层函数，可以访问外层函数的变量
        print(x)
    return inner #返回内层函数后会像胞吐一样去带着环境的变量一起返回

# 尝试直接获取 x 会报错
# print(x)  # NameError: name 'x' is not defined

# 获取闭包
closure = outer()
closure()  # 输出: 10

那么这个闭包还能怎么样呢：

In [ ]:
def outer():
    x = 10
    def inner(y):
        print(x + y)
    return inner

closure = outer() #在这一步里面closure就等于了带着x=10的inner函数
closure(5)  # 这里就相当于调用inner(5)并且已知x=10
# 输出: 15

### 这是一段来自试用期前端第四课的ppt，我们可以在其中发现一样的东西：

![alt text](image.png)

2.  **最简单的装饰器**

In [ ]:
import time

def timer_decorator(original_function):
    def wrapper(): # 内层函数，这就是将要返回的“增强版”函数
        start = time.time()
        original_function() # 执行原始函数
        end = time.time()
        print(f"{original_function.__name__} took {end - start:.2f} seconds")
    return wrapper

def say_hello():
    time.sleep(1)
    print("Hello!")

# “装饰” say_hello 函数
decorated_hello = timer_decorator(say_hello)

# 调用装饰后的函数
decorated_hello() 
# 输出: 
# Hello!
# say_hello took 1.00 seconds

上面的 `timer_decorator` 就是一个装饰器。它接受 `say_hello`，返回了 `wrapper`。调用 `decorated_hello()` 实际上是在调用 `wrapper()`。

3.  **使用 `@` 语法糖**

    Python 提供了 `@` 语法糖，让我们能更方便地使用装饰器。

 `@` 语法糖的语法：

`@装饰器函数名`

`def 函数名(参数):`

`    # 函数体`

**等价于：**

`def 函数名(参数):`

`    # 函数体`

`函数名 = 装饰器函数名(函数名)`

In [ ]:
@timer_decorator       #调用timer这个装饰器然后定义函数，就可以直接让这个函数已经是被timer装饰过的了
def say_goodbye():
    time.sleep(1.5)
    print("Goodbye!")

say_goodbye() # 直接调用即可，它已经被装饰了
# 没被装饰的时候会输出：
# Goodbye!
# 现在会输出：
# Goodbye!
# say_goodbye took 1.50 seconds

# 上面的 @timer_decorator 等效于:
# say_goodbye = timer_decorator(say_goodbye)

4.  **处理带参数的函数**
    如果原始函数有参数怎么办？我们需要让 `wrapper` 也能接收任意参数，并传递给原始函数。

In [ ]:
def timer_decorator_pro(original_function):
    def wrapper(*args, **kwargs): # 使用我们前面说的不定长参数： *args 和 **kwargs 接收所有参数
        start = time.time()
        result = original_function(*args, **kwargs) # 将参数传递给原始函数
        end = time.time()
        print(f"{original_function.__name__} took {end - start:.2f} seconds")
        return result # 返回原始函数的执行结果
    return wrapper

@timer_decorator_pro
def greet(name, repeat=1): # 这里有两个参数一个是名字一个是重复次数
    for _ in range(repeat):
        print(f"Hello, {name}!")
    return "Greeting complete"

message = greet("Alice", repeat=2)
print(message)

5.  **`functools.wraps` 的重要性**
    装饰器有一个小副作用：它会改变原函数的元信息（如 `__name__`）。
    `greet.__name__` 会变成 `'wrapper'`。为了解决这个问题，Python 提供了 `@functools.wraps`。

In [ ]:
import functools

def perfect_timer_decorator(original_function):
    @functools.wraps(original_function) # 复制元信息
    def wrapper(*args, **kwargs):
        # ... (和之前一样)
        start = time.time()
        result = original_function(*args, **kwargs)
        end = time.time()
        # 这个地方实际上是能访问到外部函数的名字，但如果没有复制就没有改变wrapper名字
        print(f"{original_function.__name__} took {end - start:.2f} seconds") 
        return result
    return wrapper

# 这个地方通过语法糖就让add指向了wrapper
@perfect_timer_decorator
def add(a, b):
    return a + b

print(add(10, 20))
# 所以这里输出的实际上是wrapper的名字，我们要把它改回去
print(add.__name__) 
# 没复制时输出: wrapper
# 有复制时输出: add

---

## **Part 2: 面向对象编程 (OOP) 的世界**

### **1. 故事的开始：我们为什么要“面向对象”？**

在学习一个新概念前，我们先看一个问题：**如果用代码来管理一个班级的学生信息，你会怎么做？**

一个学生有姓名、有年龄，还会学习。最开始，我们可能会这样做：

In [ ]:
# --- 方案一：使用零散的变量和函数 ---
student_name = "小明"
student_age = 20

def study(name, course):
    print(f"{name} 正在学习 {course}.")

study(student_name, "Python")

# 问题：如果再来一个学生“小红”，变量名得到处改，数据和函数是完全分离的，非常混乱。

这个方案显然不行。于是，我们想到了字典，它可以把属于同一个学生的数据“聚合”在一起：

In [ ]:
# --- 方案二：使用字典来聚合数据 ---
student1 = {
    'name': '小明',
    'age': 20
}

# 但行为（函数）依然是分离的
study(student1['name'], "Python")

方案二好了一些，但核心问题依旧存在：**代表学生“数据”的字典，和操作这个数据的“行为”`study` 函数，仍然是两件分开的东西。** 这就像电视机和遥控器，你得两样都备齐了才能看电视。

**那么，有没有一种更高级的方法，可以创造一种“智能实体”，它本身既包含了所有的数据（姓名、年龄），又天生就具备了所有相关的行为能力（学习）呢？**

答案是肯定的。这种将**数据**和**操作该数据的函数**“打包”在一起的编程思想，就是 **面向对象编程 (Object-Oriented Programming, OOP)** 的核心。

- **对象 (Object)**: 我们创造出来的那个“智能实体”，比如具体的某个学生“小明”。
- **类 (Class)**: 创造这种“智能实体”的**“蓝图”或“模具”**。

接下来，我们就从一个大家已经非常熟悉的例子入手，看看 Python 中的“对象”究竟是什么样的。

#### **从一个熟悉的例子理解类与对象：`datetime`**

在我们学习如何从零开始构建自己的类（比如 `Student` 类）之前，让我们先通过一个已经非常熟悉的工具——`datetime`，来直观地理解几个核心概念。这会让我们接下来的学习事半功倍。

In [ ]:
from datetime import datetime

# (1) “类”：datetime.datetime 就是一个类，可以理解为创建“时间对象”的蓝图或模板。
# 它本身不是一个具体的时间，而是定义了所有时间对象应该长什么样，有什么功能。
print(f"这是一个类: {datetime}") 
#当我们打印一个类的时候会直接打印出这个类的信息，格式为<class '模块.类名'>


# (2) “对象”或“实例”：我们通过调用类来创建一个具体的、真实存在的东西，这个东西就叫对象或实例。
# now 就是 datetime 类的一个实例。
now = datetime.now()
print(f"这是类的一个实例/对象: {now}")
print(f"它的类型是: {type(now)}")


---

我们会发现这两个输出出来的结果都是一样的一句话“它的类型是: <class 'datetime.datetime'>”这是为什么呢？

在python中type命令实际上是打印出这个对象所属的类，也就是说在python中变量什么的也是对象，我们打印变量a的数据类型时a也是个变量，而整型就是他的类

而直接打印类的时候，因为他就是一个类了所以会打印出他本身

类的类型是type，所以type（类）会输出type

而type（type）也是type，所以type本身也是一个类，这个是所有类的类叫元类

In [ ]:
a = 10
print(f"a 的类型是: {type(a)}")
# 输出: <class 'int'>，说明 a 是 int 类的一个实例

print(f"int 是: {int}")
print(f"int 的类型是: {type(int)}")
print(f"type 的类型是: {type(type)}") #所有类的类都是type
# type是所有类的类，这个是元类


---

In [ ]:
from datetime import datetime

now = datetime.now()
# (3) “实例属性”：对象内部存储的数据。我们可以通过点（.）来访问。
# 比如，now 这个对象包含了年、月、日等数据。
print(f"实例的'年'属性: {now.year}")
print(f"实例的'月'属性: {now.month}")
print(f"实例的'日'属性: {now.day}")

# (4) “实例方法”：对象能够执行的动作（函数）。我们同样通过点（.）来调用，但需要加上括号(),因为他是个函数。
# 比如，我们可以命令 now 这个对象，把它自己格式化成一个我们想要的字符串。
iso_format_string = now.isoformat()
print(f"调用实例的'isoformat'方法: {iso_format_string}")

# 再比如，我们可以让它把自己格式化成中文语境的字符串。
chinese_format_string = now.strftime('%Y年%m月%d日 %H:%M:%S')
print(f"调用实例的'strftime'方法: {chinese_format_string}")

**小结一下刚才看到的：**

*   **类 (Class)**: 蓝图，例如 `datetime` , `int` ,  `type`。
*   **对象/实例 (Object/Instance)**: 由蓝图创造出的实体，例如 `now` , `整数变量`。
*   **属性 (Attribute)**: 对象的特征/数据，通过 `对象.属性` 访问，例如 `now.year`。
*   **方法 (Method)**: 对象的行为/功能，通过 `对象.方法()` 调用，例如 `now.strftime()`。

带着这些直观理解，我们现在就可以开始学习如何定义自己的“蓝图”——类了。

### **2. 构建你的第一个类**

- **`class` 关键字**: 用于定义一个类。
- **`__init__(self, ...)`**: 构造方法。当创建一个对象时（如 `stu = Student(...)`），这个方法会自动被调用，用来初始化对象的属性。
- **`self`**: 代表对象实例本身。在类的方法中，必须把 `self` 作为第一个参数，通过它来访问对象的属性和调用其他方法。

In [ ]:
class Student:
    """定义了一个学生类"""
    
    # 构造方法，用于初始化
    def __init__(self, name: str, age: int):
        self.name = name # 实例属性对应now.year
        self.age = age   # 实例属性对应now.month
        
    # 实例方法
    def study(self, course_name: str):
        print(f"{self.name} is studying {course_name}.")

# 创建 Student 类的两个对象（实例）类似与now = datetime.now()
student1 = Student("Alice", 20)
student2 = Student("Bob", 21)

# 调用对象的方法
# 类似于now.isoformat()
student1.study("Python") # 输出: Alice is studying Python.
student2.study("Math")   # 输出: Bob is studying Math.

# 访问对象的属性类似于now.year
print(f"{student1.name}'s age is {student1.age}.") # 输出: Alice's age is 20.

### **3. 封装：数据的“保护壳”**

我们刚刚创建的 `Student` 类看起来不错，但其实有一个隐藏的“**安全隐患**”：

In [ ]:
student1 = Student("Alice", 20)
print(f"{student1.name} 的年龄是 {student1.age} 岁。")

# 外部代码可以随意修改对象的内部数据！
student1.age = -5 # 这在现实世界中是不可能的，但在代码中却发生了
print(f"修改后, {student1.name} 的年龄变成了 {student1.age} 岁。")

对象的内部数据（属性）可以被外部代码随意“篡改”，这非常危险。尤其是在构建大型项目时，你无法保证别人（或者未来的自己）不会写出错误的代码，赋予一个不合逻辑的值。

**封装 (Encapsulation)** 就是为了解决这个问题而生的。它的核心思想很简单：
1.  把对象的敏感数据“**隐藏**”起来，不让外部直接访问。
2.  提供一套**受控的“官方”方法**，作为外部与内部数据交互的唯一渠道。

这就好比银行的 ATM 机：你不能直接去动金库里的钱（隐藏数据），但可以通过 ATM 机的“存款”和“取款”按钮（官方方法）来安全地操作你的余额。

接下来，我们看看在 Python 中如何实现封装。

在 Python 中，封装主要通过命名约定来实现：
- `_protected`: 单下划线开头，告诉其他开发者：“这是一个内部属性，请不要在外部直接修改，但如果你非要改，也拦不住你。”
- `__private`: 双下划线开头，Python 会对其进行“名称改写”（Name Mangling），使得在外部很难直接访问（如 `obj.__name` 会变成 `obj._ClassName__name`），提供了更强的保护。

In [3]:
class BankAccount:
    """定义了一个银行账户类"""
    def __init__(self, account_holder, initial_balance):
        self.account_holder = account_holder
        self.__balance = initial_balance # 私有属性,账号余额应该是私密的，不能被外部直接访问

    def deposit(self, amount):
        # 存款方法
        if amount > 0:
            self.__balance += amount
            print(f"存入 ${amount}. 新余额: ${self.__balance}")
        else:
            print("存款金额必须是正数.")

    def withdraw(self, amount):
        # 取款方法
        if 0 < amount <= self.__balance:
            self.__balance -= amount
            print(f"取出 ${amount}. 新余额: ${self.__balance}")
        else:
            print("取款金额无效.")

    def get_balance(self):
        return self.__balance

account = BankAccount("Charlie", 1000)
# print(account.__balance) # 这会报错 AttributeError

# 只能通过提供的公共方法来操作余额
account.deposit(500)
account.withdraw(200)
print(f"当前余额: ${account.get_balance()}")

存入 $500. 新余额: $1500
取出 $200. 新余额: $1300
当前余额: $1300


#### **`@property`**

有时候我们想让一个方法的调用看起来像属性访问（即不加括号）。`@property` 装饰器可以做到这一点，它常用于创建只读属性。

In [ ]:
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def diameter(self):
        """将 diameter 方法变成一个只读属性"""
        return self.radius * 2

c = Circle(5)
print(c.radius)   # 5
print(c.diameter) # 调用时像属性，无需括号，输出: 10
# c.diameter = 20 # 这会报错，因为这个只是一个只读属性

---

### **4. 继承：代码复用的利器**

我们的学生管理系统运行得不错。现在，**新的需求来了**：我们不仅要管理学生，还要管理**老师**。

老师和学生有很多**共同点**，比如他们都有姓名和年龄，都会“说话”。但老师又有自己的**特点**，比如有“学科”属性，有“教学”行为。

最笨的办法，就是把 `Student` 类的代码复制一遍，改名叫 `Teacher`：

In [ ]:
# 冗余的 Teacher 类
class Teacher:
    def __init__(self, name, age, subject): # name 和 age 的代码是重复的
        self.name = name
        self.age = age
        self.subject = subject # 新增属性

    def speak(self): # 这个方法和学生的完全一样
        print(f"{self.name} is speaking.")

    def teach(self): # 新增方法
        print(f"{self.name} is teaching {self.subject}.")

这样做充满了“坏味道”：大量的重复代码，如果将来要给 `Person` 增加“性别”属性，就得在 `Student` 和 `Teacher` 两个类里都改一遍，非常难以维护。

**继承 (Inheritance)** 就是为了解决“**代码复用**”和“**抽象通用模式**”而生的。它的核心思想是：
1.  **提取共性**：将学生和老师的共同特征（`name`, `age`, `speak`）提取出来，定义一个更通用的“父类”，比如 `Person`。
2.  **继承并扩展**：让 `Student` 和 `Teacher` 类都**继承**这个 `Person` 类，这样它们就自动拥有了 `Person` 的所有属性和方法。然后，它们只需要专注于编写自己**独有**的“扩展”部分即可。

- **`super()`**: 在子类中，可以使用 `super()` 来调用父类的方法，最常见于 `__init__` 中。
- **方法重写 (Overriding)**: 如果子类定义了一个和父类同名的方法，那么子类的方法会覆盖父类的方法。

In [ ]:
# 父类
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def speak(self):
        print(f"{self.name} is speaking.")

# 子类，继承自 Person
class Teacher(Person):
    def __init__(self, name, age, subject):
        super().__init__(name, age)  # 调用父类的构造方法来初始化 name 和 age
        self.subject = subject  # 添加自己的属性

    # 重写父类的 speak 方法
    def speak(self):
        print(f"{self.name} is a teacher and is speaking about {self.subject}.")

    # 添加自己的方法
    def teach(self):
        print(f"{self.name} is teaching.")

p = Person("David", 45)
t = Teacher("Emily", 35, "Physics")

p.speak()  # David is speaking.
t.speak()  # Emily is a teacher and is speaking about Physics. (调用了重写后的方法)
t.teach()  # Emily is teaching.

### **5. 多态：一种接口，多种形态**

我们的系统越来越完善了，现在有 `Student` 类和 `Teacher` 类。**新的需求又来了**：我们需要举办一次全校大会，要求所有参会人员（无论是学生还是老师）都进行“签到”。

最直接的想法可能是写一个函数，用 `if/elif` 来判断对象的具体类型：

In [ ]:
def do_sign_in(person):
    if isinstance(person, Teacher):
        print(f"老师 {person.name} 已签到，请到教师席就坐。")
    elif isinstance(person, Student):
        print(f"同学 {person.name} 已签到，请按班级就坐。")
    # ... 如果将来还有辅导员、校长等角色，这里的if/elif会无限延长！

这个函数非常“僵化”。每增加一种新角色，我们都必须修改 `do_sign_in` 函数的内部代码，这违反了“开放封闭原则”（对扩展开放，对修改封闭）。

**多态 (Polymorphism)** 提供了一种更优雅的解决方案。它的核心思想是：**“只管发号施令，具体怎么做由对象自己决定”**。

我们不关心一个对象到底是 `Student` 还是 `Teacher`，我们只管统一调用它的 `.sign_in()` 方法。至于学生和老师如何执行签到这个动作，是他们各自内部的事情。

这种“**一种接口，多种形态**”的特性，就是多态。它极大地提高了代码的灵活性和可扩展性。

In [ ]:
class Animal:
    def make_sound(self):
        raise NotImplementedError # 强制子类必须实现这个方法否则就会报错

class Dog(Animal):
    def make_sound(self):
        return "Woof!"

class Cat(Animal):
    def make_sound(self):
        return "哈！"

class CheesyLeopard(Animal):
    def make_sound(self):
        return "芝士雪豹！"

def animal_sound(animals: list[Animal]):
    for animal in animals:
        # 我们不关心 animal 到底是 Dog、Cat 还是 CheesyLeopard
        # 只管调用 make_sound 方法，对象自己会决定如何响应
        print(animal.make_sound())

dog = Dog()
cat = Cat()
cheesy_leopard = CheesyLeopard()
animal_sound([dog, cat, cheesy_leopard])
# 输出:
# Woof!
# 哈！
# 芝士雪豹！

`animal_sound` 函数就是多态的体现。它接受一个动物列表，并对每个动物调用 `make_sound()`。由于每个子类都以自己的方式实现了这个方法，所以我们听到了不同的叫声。

---

### **6. 类的“高级技能”：让类更强大**

到目前为止，我们定义的 `study` 方法，它的第一个参数是 `self`，我们称之为“**实例方法**”。这种方法是和一个具体的学生**实例**绑定的。

但有时候，我们会遇到一些新的问题，需要一些不依赖于任何具体实例的“特殊方法”。

#### **`@classmethod`：创建实例的“备用途径”**

**新问题**：我们的 `Student` 类现在通过 `Student("Alice", 20)` 的方式来创建，非常方便。但现在，教务处给了一份用连字符连接的字符串名单，格式是 `'姓名-年龄'`，比如 `'Bob-21'`。我们如何根据这种特殊的字符串，来创建 `Student` 实例呢？

一个直接的想法是在类的外部写一个辅助函数：

In [ ]:
def create_student_from_string(text):
    name, age_str = text.split('-')
    # 注意，这里写死了 Student 类名
    return Student(name, int(age_str))

s2 = create_student_from_string('Bob-21')
print(f"从字符串创建的学生: {s2.name}, {s2.age}岁")

这样做可以，但感觉很“别扭”。`create_student_from_string` 这个函数明明是为 `Student` 类量身定做的，却要“流落在外”，破坏了我们面向对象“打包”的初衷。

**解决方案**：我们希望 `Student` **类自己**就具备“从字符串创建实例”的能力。`@classmethod` 就是为此而生的。

它定义的“**类方法**”，第一个参数不再是 `self`（某个具体实例），而是 `cls`（**类本身**）。这使得我们可以在方法内部，通过 `cls(...)` 这种灵活的方式来调用类的构造器，创建实例。

In [ ]:
class Student:
    def __init__(self, name: str, age: int):
        self.name = name
        self.age = age

    def study(self, course_name: str):
        print(f"{self.name} is studying {course_name}.")

    @classmethod
    def from_string(cls, student_string: str):
        """
        这是一个类方法, 它提供了一种从特定格式字符串创建实例的“备用途径”。
        cls 参数由 Python 自动传入，它代表 Student 这个类本身。
        """
        name, age_str = student_string.split('-')
        age = int(age_str)
        # 调用 cls() 就等同于调用 Student()，但这样做更灵活
        # 如果将来类改名为 SuperStudent，这里不需要任何修改
        return cls(name, age)

# 现在，创建方式变得非常优雅和直观！
s3 = Student.from_string('Charlie-22')
print(f"用类方法创建的学生: {s3.name}, {s3.age}岁")

**小结**：`@classmethod` 最核心的用途就是作为“**工厂方法**”，为创建类的实例提供多种不同的途径。

#### **`@staticmethod`：寄宿在类中的“工具函数”**

**新问题**：在创建学生或修改年龄时，我们可能想增加一个验证逻辑，比如“年龄必须在 6 到 80 岁之间”。

同样，我们可以在外部写一个独立的工具函数 `is_valid_age(age)`。但这个函数几乎只为 `Student` 类服务，让它在全局“飘着”，有点污染命名空间，不够整洁。我们希望把它“收纳”到 `Student` 类里面，仅仅是为了更好地组织代码。

**解决方案**：使用 `@staticmethod`。

它定义的“**静态方法**”，就是一个**不依赖任何外部状态的、普通的函数**，它只是“寄宿”在类的命名空间里。
*   它**不关心**某个具体的学生实例（所以不需要 `self`）。
*   它也**不关心** `Student` 这个类蓝图（所以不需要 `cls`）。

In [ ]:
class Student:
    def __init__(self, name: str, age: int):
        # 在初始化时就使用静态方法进行验证
        if not Student.is_valid_age(age):
            raise ValueError("无效的年龄！")
        self.name = name
        self.age = age
    
    # ... 其他方法 ...

    @staticmethod
    def is_valid_age(age: int) -> bool:
        """
        这是一个静态方法。它就是一个功能独立的工具函数，
        只是为了组织代码的方便，将它放在了 Student 类内部。
        """
        return 6 <= age <= 80

# 现在，我们可以在类的内部和外部方便地使用它
if Student.is_valid_age(25):
    s4 = Student("David", 25)
    print("学生 David 创建成功。")

# 尝试创建一个无效年龄的学生
try:
    s5 = Student("Eve", 5)
except ValueError as e:
    print(f"创建学生 Eve 失败: {e}")


**`@classmethod` vs. `@staticmethod` 总结**

| 特征 | 实例方法 | `@classmethod` (类方法) | `@staticmethod` (静态方法) |
| :--- | :--- | :--- | :--- |
| **第一个参数** | `self` (实例对象) | `cls` (类本身) | 无特殊参数 |
| **核心用途** | 操作**实例**的属性和状态 | 作为**工厂方法**，创建实例 | **工具函数**，与类和实例都无关 |
| **调用方式** | `实例.方法()` | `类.方法()` | `类.方法()` |
| **一句话比喻** | “我的行为” | “我们家族的能力” | “挂在家族墙上的一个工具” |

### **7. 终极技能：运算符重载 —— 让你的对象符合直觉**

**新问题**：我们创建了 `Student` 对象后，会遇到一些“反直觉”的尴尬时刻。

1.  **无法友好地打印**：`print(student1)` 输出的是一串难懂的内存地址 `<__main__.Student object at ...>`，而不是我们期望的 “学生: Alice, 20岁”。
2.  **无法直接比较**：`if student2 > student1:` 会直接报错 `TypeError`，因为 Python 不知道该如何比较两个 `Student` 对象的大小。我们只能写 `if student2.age > student1.age:`，不够优雅。

**解决方案**：Python 提供了一套“魔法方法”（以双下划线开头和结尾），通过实现它们，我们就能“**教会**”我们自己创造的类如何响应标准的运算符（如 `>`、`+`）和内置函数（如 `print()`）。这个过程，就叫做**运算符重载 (Operator Overloading)**。

它的核心就是：**让你的自定义对象，也能像 `int`、`str` 那样，用最自然的方式去操作。**

#### **让对象“会自我介绍”：重载 `__str__`**

`__str__` 方法会在我们使用 `print()` 函数或者 `str()` 类型转换函数作用于一个对象时被自动调用。它必须返回一个字符串。

In [ ]:
class Student:
    def __init__(self, name: str, age: int):
        self.name = name
        self.age = age

    def __str__(self) -> str:
        """定义当 print(student) 时应该显示什么"""
        return f"学生: {self.name}, {self.age}岁"

s1 = Student("Alice", 20)
print(s1) # 不再是内存地址，而是调用了 __str__ 的结果
# 输出: 学生: Alice, 20岁

#### **让对象“可比较大小”：重载 `__lt__`, `__gt__` 等**

`__lt__` (less than) 定义了 `<` 的行为，`__gt__` (greater than) 定义了 `>` 的行为。我们来实现一下按年龄比较大小。

In [ ]:
class Student:
    def __init__(self, name: str, age: int):
        self.name = name
        self.age = age

    def __str__(self) -> str:
        return f"学生: {self.name}, {self.age}岁"

    def __gt__(self, other) -> bool:
        """定义 > 运算符的行为：比较年龄"""
        # self 代表 > 左边的对象, other 代表右边的对象
        return self.age > other.age

s1 = Student("Alice", 20)
s2 = Student("Bob", 21)

if s2 > s1:
    print(f"{s2.name} 的年龄比 {s1.name} 大。")
# 输出: Bob 的年龄比 Alice 大。

students = [s2, s1]
students.sort() # 因为定义了比较方法，所以现在列表知道该如何排序了！
print([str(s) for s in students])
# 输出: ['学生: Alice, 20岁', '学生: Bob, 21岁']

通过实现这些魔法方法，我们让我们创造的 `Student` 类变得更加强大、易用且符合编程直觉。

---

## **附录与展望**

### **1. 更强大的数据校验：Pydantic**

还记得我们用 `@staticmethod` 实现的 `is_valid_age` 吗？虽然它能工作，但在真实项目中，数据验证通常更复杂。比如，我们可能还需要验证姓名不能为空、年龄必须是整数等。

未来，我们会学习一个强大的库 `Pydantic`，它可以让我们用非常声明式的方式来定义数据模型，并自动完成数据校验和类型转换。

In [ ]:
# 预告：未来我们会学到的 Pydantic 写法
from pydantic import BaseModel, Field

class StudentModel(BaseModel):
    name: str = Field(min_length=1)
    age: int = Field(gt=5, lt=81) # gt: greater than, lt: less than

# Pydantic 会自动完成验证
try:
    s = StudentModel(name="Alice", age=20)
    print("Pydantic 模型创建成功!")
    # s_invalid = StudentModel(name="Eve", age=5) # 这行代码会直接抛出 ValidationError
except Exception as e:
    print(e)

从 `Student` 类限制年龄的例子引入 Pydantic `BaseModel`，是展示从手动验证到框架自动验证演进的一个绝佳案例。

### **2. 虚拟环境管理**

在课程中我们提到了虚拟环境，这里补充一些常用的管理命令。

**删除虚拟环境**：
如果你的虚拟环境文件夹是 `.venv`，在 PowerShell 中可以使用：

In [ ]:
%%powershell
# -Recurse 表示递归删除，-Force 表示强制删除，无需确认
Remove-Item -Recurse -Force .\.venv

在 macOS 或 Linux 的 bash/zsh 中，或者 Windows 的 Git Bash 中，可以使用：

In [ ]:
%%bash
# rm -r 表示递归删除, -f 表示强制
rm -rf .venv

> **小技巧**：在命令行中，你可以输入 `Remove-` 或 `rm` 然后按 `Tab` 键，Shell 会自动帮你补全命令或路径，非常方便！

### **3. hyh的私货：全局虚拟环境管理器 `vu`**
对于习惯了 `conda` 那样全局管理所有环境的同学，这有一个小脚本 `vu`，它模仿了 `conda` 的常用命令（如 `vu create`, `vu activate`, `vu ls`, `vu rm`），但底层使用的是 `uv`。如果你感兴趣，可以从这里找到它：
[https://github.com/zeroHYH/vu](https://github.com/zeroHYH/vu)